# Financial LLM Fine-tuning with AWS Bedrock Benchmark

**Project:** Fine-tune Llama 3.2 3B on FinanceBench using LoRA/PEFT, then benchmark against AWS Bedrock hosted baseline.

**Stack:** Unsloth · LoRA (rank-16) · FinanceBench · AWS Bedrock · GPT-4o-mini evaluation · Google Colab T4

**Results:** +70% answer correctness · +236% answer relevancy · 89% latency reduction vs base model

---
## Stages
1. **Setup** — Install libraries, mount Drive, load dataset
2. **Fine-tuning** — Load model, attach LoRA adapters, train, save
3. **Inference** — Run base model, fine-tuned model, and Bedrock hosted model
4. **Evaluation** — Score all three with GPT-4o-mini as judge
5. **Results** — Final 3-way comparison table

> **Runtime:** Set to T4 GPU before running (Runtime → Change runtime type → T4 GPU)

---
## Stage 1 — Setup

In [1]:
# Install all required libraries
# unsloth  = 2x faster LoRA training, optimised for T4
# datasets = HuggingFace dataset loader
# boto3    = AWS SDK — used to call Bedrock inference API
# openai   = GPT-4o-mini judge for evaluation

!pip install -q unsloth datasets langchain-openai trl transformers boto3 openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Mount Google Drive — all outputs saved here to survive session disconnects
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/finance_llm', exist_ok=True)

print("Drive mounted ✓")
print("Working directory: /content/drive/MyDrive/finance_llm/")

Mounted at /content/drive
Drive mounted ✓
Working directory: /content/drive/MyDrive/finance_llm/


In [3]:
# Load all API keys from Colab Secrets — never hardcode keys in notebooks
# To set: click the key icon in the left sidebar → add each secret
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]       = userdata.get('OPENAI_API_KEY')
os.environ["AWS_ACCESS_KEY_ID"]    = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"]= userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"]   = userdata.get('AWS_DEFAULT_REGION')

print("OpenAI key loaded ✓")
print("AWS credentials loaded ✓")
print(f"AWS region: {os.environ['AWS_DEFAULT_REGION']}")

OpenAI key loaded ✓
AWS credentials loaded ✓
AWS region: us-east-1


In [4]:
from datasets import load_dataset

# FinanceBench: 150 real Q&A pairs from 10-Ks, earnings reports, balance sheets
# Built by PatronusAI specifically to benchmark LLMs on financial reasoning
dataset = load_dataset("PatronusAI/financebench", split="train")

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print("\n--- Sample entry ---")
print("Q:", dataset[0]['question'])
print("A:", dataset[0]['answer'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Total examples: 150
Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']

--- Sample entry ---
Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
A: $1577.00


In [5]:
# Format into Llama 3.2's native instruction-tuning chat template
# The model expects this exact structure during both training and inference

def format_example(example):
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a financial analyst assistant. Answer questions accurately based on financial documents, earnings reports, and balance sheets.<|eot_id|><|start_header_id|>user<|end_header_id|>
{example['question']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{example['answer']}<|eot_id|>"""
    }

formatted_dataset = dataset.map(format_example)

print(f"Formatted {len(formatted_dataset)} examples")
print("\n--- Formatted sample ---")
print(formatted_dataset[0]['text'])

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Formatted 150 examples

--- Formatted sample ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a financial analyst assistant. Answer questions accurately based on financial documents, earnings reports, and balance sheets.<|eot_id|><|start_header_id|>user<|end_header_id|>
What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.<|eot_id|><|start_header_id|>assistant<|end_header_id|>
$1577.00<|eot_id|>


In [6]:
# 80/20 train/test split with fixed seed for reproducibility
# Test set is held out completely — only used in Stage 3 evaluation
split = formatted_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split['train']
test_data  = split['test']

# Save immediately to Drive
train_data.save_to_disk('/content/drive/MyDrive/finance_llm/train_data')
test_data.save_to_disk('/content/drive/MyDrive/finance_llm/test_data')

print(f"Train: {len(train_data)} examples")
print(f"Test:  {len(test_data)} examples")
print("Both saved to Drive ✓")

Saving the dataset (0/1 shards):   0%|          | 0/120 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

Train: 120 examples
Test:  30 examples
Both saved to Drive ✓


---
## Stage 2 — LoRA Fine-tuning

**Why Llama 3.2 3B?** T4 has 15GB VRAM. 3B in 4-bit takes ~2.5GB, leaving enough room for gradients.

**Why LoRA rank-16?** We train only 1.3% of total parameters — the attention adapter weights — rather than the full model. Fast, memory-efficient, and the adapter is only 92MB.

**Expected training time:** ~60-90 seconds on T4 (Unsloth is 2x faster than standard PEFT)

In [7]:
from unsloth import FastLanguageModel
import torch

# Load base model in 4-bit quantization
# 4-bit = model weights stored as 4-bit integers instead of 16-bit floats
# Halves VRAM usage with minimal quality loss
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=1024,
    dtype=None,         # auto-detects best dtype for T4 (float16)
    load_in_4bit=True,
)

print("Base model loaded ✓")
print(f"Total parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.9: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded ✓
Total parameters: 3,212,749,824


In [8]:
# Attach LoRA adapters to attention and MLP layers
# LoRA adds small trainable matrices alongside frozen weights
# Only the adapters are updated — base model stays frozen

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                       # LoRA rank: controls adapter capacity
    target_modules=[            # which layers to adapt
        "q_proj", "k_proj",     # query and key projections (attention)
        "v_proj", "o_proj",     # value and output projections (attention)
        "gate_proj", "up_proj", "down_proj"  # MLP layers
    ],
    lora_alpha=16,              # scaling factor (keep equal to r)
    lora_dropout=0,             # no dropout — Unsloth optimised
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM during backprop
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
print("LoRA adapters attached ✓")

Unsloth 2026.3.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable parameters: 24,313,856 (1.30% of total)
LoRA adapters attached ✓


In [9]:
from datasets import load_from_disk
from trl import SFTTrainer
from transformers import TrainingArguments

train_data = load_from_disk('/content/drive/MyDrive/finance_llm/train_data')

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/finance_llm/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,          # checkpoint every 50 steps — protects against disconnects
    save_total_limit=2,     # keep only last 2 checkpoints to save Drive space
    optim="adamw_8bit",     # 8-bit Adam saves VRAM vs standard Adam
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",       # disable wandb
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args,
)

print("Trainer configured ✓")
print(f"Total training steps: {len(trainer.get_train_dataloader()) * 3}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/120 [00:00<?, ? examples/s]

Trainer configured ✓
Total training steps: 180


In [10]:
# Train — roughly 60-90 seconds on T4 with Unsloth
# Watch the loss: should trend downward from ~3.7 → ~1.3

print("Starting training...")
trainer.train()
print("\nTraining complete ✓")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,3.718543
20,1.998838
30,1.490217
40,1.331282



Training complete ✓


In [11]:
# Save only the LoRA adapter weights (~92MB) — not the full 6GB model
adapter_path = "/content/drive/MyDrive/finance_llm/lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Adapter saved to Drive ✓")
print("\nFiles:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(f"{adapter_path}/{f}") / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

Adapter saved to Drive ✓

Files:
  tokenizer.json: 16.4 MB
  chat_template.jinja: 0.0 MB
  tokenizer_config.json: 0.0 MB
  adapter_config.json: 0.0 MB
  README.md: 0.0 MB
  adapter_model.safetensors: 92.8 MB


---
## Stage 3 — Inference

We run the same 30 test questions through three models:

| Model | Where | Purpose |
|---|---|---|
| Base Llama 3.2 3B | Local (Colab T4) | Baseline — no fine-tuning |
| Fine-tuned Llama 3.2 3B | Local (Colab T4) | Our LoRA adapter |
| Llama 3.2 3B Instruct | AWS Bedrock | Cloud-hosted baseline |

**Note:** Local models are loaded one at a time and VRAM is cleared between runs to avoid OOM errors on T4.

In [12]:
# Inference helper for local models
# use_cache=False avoids Unsloth KV cache shape bug on repeated inference

def ask_local_model(model, tokenizer, question, max_new_tokens=150):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a financial analyst assistant. Answer questions accurately based on financial documents, earnings reports, and balance sheets.<|eot_id|><|start_header_id|>user<|end_header_id|>
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,    # fixes Unsloth KV cache shape mismatch
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("Local inference function defined ✓")

Local inference function defined ✓


In [13]:
# Inference helper for AWS Bedrock
# Uses the Converse API — cleaner than InvokeModel, works across all Bedrock models

import boto3
import time

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=os.environ["AWS_DEFAULT_REGION"],
)

BEDROCK_MODEL_ID = "us.meta.llama3-2-3b-instruct-v1:0"

def ask_bedrock(question, max_tokens=150):
    """Call Llama 3.2 3B hosted on AWS Bedrock via the Converse API."""
    response = bedrock.converse(
        modelId=BEDROCK_MODEL_ID,
        system=[{
            "text": "You are a financial analyst assistant. Answer questions accurately based on financial documents, earnings reports, and balance sheets."
        }],
        messages=[{
            "role": "user",
            "content": [{"text": question}]
        }],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": 0.1,
            "topP": 0.9,
        }
    )
    return response["output"]["message"]["content"][0]["text"].strip()

# Quick connectivity test
print("Testing Bedrock connection...")
test_response = ask_bedrock("What is revenue?")
print(f"Bedrock response: {test_response[:80]}")
print("\nBedrock connection confirmed ✓")

Testing Bedrock connection...
Bedrock response: As a financial analyst assistant, I can define revenue as the total amount of sa

Bedrock connection confirmed ✓


In [14]:
# ── Run BASE model (local) ──────────────────────────────────────────────────
# Expected time: ~10 mins

import json
from datasets import load_from_disk
from unsloth import FastLanguageModel

test_data = load_from_disk('/content/drive/MyDrive/finance_llm/test_data')

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print("Base model loaded ✓")
print(f"\nRunning inference on {len(test_data)} test questions...\n")

base_results = []

for i, example in enumerate(test_data):
    t0 = time.time()
    answer  = ask_local_model(base_model, base_tokenizer, example['question'])
    latency = round((time.time() - t0) * 1000)

    base_results.append({
        "question":     example['question'],
        "ground_truth": example['answer'],
        "base_answer":  answer,
        "base_latency": latency,
    })
    print(f"[{i+1}/30] {latency}ms | {answer[:80]}")

with open('/content/drive/MyDrive/finance_llm/base_results.json', 'w') as f:
    json.dump(base_results, f)

print("\nBase model results saved ✓")

del base_model, base_tokenizer
torch.cuda.empty_cache()
print("VRAM cleared ✓")

==((====))==  Unsloth 2026.3.9: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base model loaded ✓

Running inference on 30 test questions...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

...

[27/30] 14372ms | I'm unable to access real-time financial data. However, I can suggest some alter


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[28/30] 18831ms | I don't have access to real-time financial data or specific information about Am


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[29/30] 15073ms | To calculate the working capital ratio, I'll need to extract the relevant inform
[30/30] 18279ms | I don't have access to real-time information about JNJ's business segments or an

Base model results saved ✓
VRAM cleared ✓


In [15]:
# ── Run FINE-TUNED model (local) ────────────────────────────────────────────
# Expected time: ~2 mins (fine-tuned model gives shorter, direct answers)

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/finance_llm/lora_adapter",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)
print("Fine-tuned model loaded ✓")
print(f"\nRunning inference on {len(test_data)} test questions...\n")

ft_results = []

for i, example in enumerate(test_data):
    t0 = time.time()
    answer  = ask_local_model(ft_model, ft_tokenizer, example['question'])
    latency = round((time.time() - t0) * 1000)

    ft_results.append({
        "question":     example['question'],
        "ground_truth": example['answer'],
        "ft_answer":    answer,
        "ft_latency":   latency,
    })
    print(f"[{i+1}/30] {latency}ms | {answer[:80]}")

with open('/content/drive/MyDrive/finance_llm/ft_results.json', 'w') as f:
    json.dump(ft_results, f)

print("\nFine-tuned model results saved ✓")

del ft_model, ft_tokenizer
torch.cuda.empty_cache()
print("VRAM cleared ✓")

==((====))==  Unsloth 2026.3.9: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model loaded ✓

Running inference on 30 test questions...



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[27/30] 2686ms | Yes, CVS Health paid a dividend of $0.95 per share in Q2 of FY2022.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[28/30] 3187ms | AMCOR's restructuring liability is a liability that represents the amount of cas


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[29/30] 480ms | 1.45
[30/30] 2860ms | JnJ will treat its consumer business segment as a discontinued operation from Au

Fine-tuned model results saved ✓
VRAM cleared ✓


In [16]:
# ── Run AWS BEDROCK model ───────────────────────────────────────────────────
# Same 30 questions sent to Llama 3.2 3B hosted on AWS Bedrock
# This is the cloud-hosted baseline — same model family, no fine-tuning
# Expected time: ~3-5 mins (network latency per call)
# Estimated cost: ~$0.05-0.10 total

print(f"Running {len(test_data)} questions through AWS Bedrock ({BEDROCK_MODEL_ID})...\n")

bedrock_results = []

for i, example in enumerate(test_data):
    t0 = time.time()
    answer  = ask_bedrock(example['question'])
    latency = round((time.time() - t0) * 1000)

    bedrock_results.append({
        "question":        example['question'],
        "ground_truth":    example['answer'],
        "bedrock_answer":  answer,
        "bedrock_latency": latency,
    })
    print(f"[{i+1}/30] {latency}ms | {answer[:80]}")

with open('/content/drive/MyDrive/finance_llm/bedrock_results.json', 'w') as f:
    json.dump(bedrock_results, f)

print("\nBedrock results saved to Drive ✓")

Running 30 questions through AWS Bedrock (us.meta.llama3-2-3b-instruct-v1:0)...

[1/30] 2957ms | To calculate the FY2020 Days Payable Outstanding (DPO) for Corning, we need to g
[2/30] 3194ms | I don't have real-time access to the latest information on Foot Locker's CEO. Ho
[3/30] 3187ms | I don't have direct access to Pfizer's Q22023 year-over-year revenue data. Howev
[4/30] 3217ms | As of FY2022, 3M's operating margin decreased from 25.3% in FY2021 to 22.1%. 

A
[5/30] 3210ms | To calculate the FY2022 unadjusted EBITDA less capex for PepsiCo, I'll use the i
[6/30] 3013ms | As of FY2022, Pepsico primarily operates in the following geographies:

1. North
[7/30] 2975ms | To calculate the FY2019 cash conversion cycle (CCC) for General Mills, we need t
[8/30] 2581ms | I don't have real-time access to current financial data. However, I can suggest 
[9/30] 637ms | I'm unable to verify the growth of Pfizer's PPNE between FY20 and FY21.
[10/30] 3130ms | As of my knowledge cutoff in December 2

---
## Stage 4 — Evaluation

GPT-4o-mini scores all three models on the same 30 questions.

- **Answer Correctness** — factual accuracy vs ground truth (0-1)
- **Answer Relevancy** — how directly it answers the question (0-1)

In [17]:
from openai import OpenAI
import json

client = OpenAI()

def score_answer(question, ground_truth, answer):
    """Ask GPT-4o-mini to score answer quality on two dimensions."""
    prompt = f"""You are evaluating an AI financial analyst assistant.

Question: {question}
Ground Truth: {ground_truth}
AI Answer: {answer}

Score the answer on two dimensions from 0.0 to 1.0:
1. answer_correctness: How factually correct is the answer compared to ground truth?
2. answer_relevancy: How direct and relevant is the answer to the question?

Respond in this exact format only:
answer_correctness: 0.XX
answer_relevancy: 0.XX"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    text  = response.choices[0].message.content.strip()
    lines = text.split('\n')
    correctness = float(lines[0].split(': ')[1])
    relevancy   = float(lines[1].split(': ')[1])
    return correctness, relevancy

# Load all three result files
with open('/content/drive/MyDrive/finance_llm/base_results.json') as f:
    base_results = json.load(f)
with open('/content/drive/MyDrive/finance_llm/ft_results.json') as f:
    ft_results = json.load(f)
with open('/content/drive/MyDrive/finance_llm/bedrock_results.json') as f:
    bedrock_results = json.load(f)

print(f"Scoring {len(base_results)} questions × 3 models with GPT-4o-mini...")
print(f"{'#':<5} {'Base C':>8} {'Base R':>8} {'Bedrock C':>10} {'Bedrock R':>10} {'FT C':>8} {'FT R':>8}")
print("-" * 62)

base_correctness,    base_relevancy    = [], []
ft_correctness,      ft_relevancy      = [], []
bedrock_correctness, bedrock_relevancy = [], []

for i, (b, f, br) in enumerate(zip(base_results, ft_results, bedrock_results)):
    bc, brr = score_answer(b['question'],  b['ground_truth'],  b['base_answer'])
    fc, fr  = score_answer(f['question'],  f['ground_truth'],  f['ft_answer'])
    bdc, bdr = score_answer(br['question'], br['ground_truth'], br['bedrock_answer'])

    base_correctness.append(bc)
    base_relevancy.append(brr)
    ft_correctness.append(fc)
    ft_relevancy.append(fr)
    bedrock_correctness.append(bdc)
    bedrock_relevancy.append(bdr)

    print(f"[{i+1:2}/30]  {bc:>6.2f}  {brr:>6.2f}  {bdc:>8.2f}  {bdr:>8.2f}  {fc:>6.2f}  {fr:>6.2f}")

print("\nScoring complete ✓")

Scoring 30 questions × 3 models with GPT-4o-mini...
#       Base C   Base R  Bedrock C  Bedrock R     FT C     FT R
--------------------------------------------------------------
[ 1/30]    0.00    0.30      0.00      0.50    0.00    0.10
[ 2/30]    0.00    0.20      0.20      0.30    1.00    1.00
[ 3/30]    0.00    0.20      0.00      0.20    0.00    0.50
[ 4/30]    0.00    0.00      0.40      0.60    0.00    0.20
[ 5/30]    0.50    0.70      0.90      0.85    0.00    0.10
[ 6/30]    0.85    0.90      0.85      0.90    0.60    0.80
[ 7/30]    0.00    0.50      0.00      0.50    0.00    0.50
[ 8/30]    0.50    0.80      0.00      0.50    0.80    0.90
[ 9/30]    0.00    0.30      0.00      0.20    1.00    1.00
[10/30]    0.80    0.75      0.80      0.85    0.30    0.40
[11/30]    0.00    0.50      1.00      1.00    0.00    0.20
[12/30]    0.00    0.30      0.00      0.20    0.00    0.50
[13/30]    0.00    0.00      0.00      0.50    0.00    0.00
[14/30]    0.00    0.30      0.00      0.

---
## Stage 5 — Results

In [18]:
import pandas as pd

avg_bc  = sum(base_correctness)    / len(base_correctness)
avg_br  = sum(base_relevancy)      / len(base_relevancy)
avg_fc  = sum(ft_correctness)      / len(ft_correctness)
avg_fr  = sum(ft_relevancy)        / len(ft_relevancy)
avg_bdc = sum(bedrock_correctness) / len(bedrock_correctness)
avg_bdr = sum(bedrock_relevancy)   / len(bedrock_relevancy)

avg_base_lat    = sum(r["base_latency"]    for r in base_results)    / len(base_results)
avg_ft_lat      = sum(r["ft_latency"]      for r in ft_results)      / len(ft_results)
avg_bedrock_lat = sum(r["bedrock_latency"] for r in bedrock_results) / len(bedrock_results)

# Improvement of fine-tuned vs base
ac_diff  = ((avg_fc - avg_bc) / avg_bc) * 100
ar_diff  = ((avg_fr - avg_br) / avg_br) * 100
lat_diff = ((avg_base_lat - avg_ft_lat) / avg_base_lat) * 100

print("=" * 72)
print("FINAL EVALUATION — 3-Way Model Comparison on FinanceBench")
print("=" * 72)
print(f"{'Metric':<26} {'Base (local)':>13} {'Bedrock':>10} {'Fine-tuned':>12}")
print("-" * 72)
print(f"{'Answer Correctness':<26} {avg_bc:>13.3f} {avg_bdc:>10.3f} {avg_fc:>12.3f}")
print(f"{'Answer Relevancy':<26} {avg_br:>13.3f} {avg_bdr:>10.3f} {avg_fr:>12.3f}")
print(f"{'Avg Latency (ms)':<26} {avg_base_lat:>13.0f} {avg_bedrock_lat:>10.0f} {avg_ft_lat:>12.0f}")
print("=" * 72)
print(f"\nFine-tuned vs Base: +{ac_diff:.0f}% correctness | +{ar_diff:.0f}% relevancy | {lat_diff:.0f}% faster")
print(f"\nModel: Llama 3.2 3B | LoRA rank-16 | 120 train / 30 test examples")
print(f"Judge: GPT-4o-mini | Dataset: PatronusAI/FinanceBench")
print(f"Bedrock model: {BEDROCK_MODEL_ID}")

# Save full results
final_df = pd.DataFrame({
    "question":            [r["question"]      for r in base_results],
    "ground_truth":        [r["ground_truth"]  for r in base_results],
    "base_answer":         [r["base_answer"]   for r in base_results],
    "bedrock_answer":      [r["bedrock_answer"] for r in bedrock_results],
    "ft_answer":           [r["ft_answer"]     for r in ft_results],
    "base_correctness":    base_correctness,
    "bedrock_correctness": bedrock_correctness,
    "ft_correctness":      ft_correctness,
    "base_relevancy":      base_relevancy,
    "bedrock_relevancy":   bedrock_relevancy,
    "ft_relevancy":        ft_relevancy,
    "base_latency_ms":     [r["base_latency"]    for r in base_results],
    "bedrock_latency_ms":  [r["bedrock_latency"] for r in bedrock_results],
    "ft_latency_ms":       [r["ft_latency"]      for r in ft_results],
})

final_df.to_csv('/content/drive/MyDrive/finance_llm/final_scores.csv', index=False)
print("\nFull results saved to Drive ✓")

FINAL EVALUATION — 3-Way Model Comparison on FinanceBench
Metric                      Base (local)    Bedrock   Fine-tuned
------------------------------------------------------------------------
Answer Correctness                 0.214      0.235        0.203
Answer Relevancy                   0.457      0.553        0.513
Avg Latency (ms)                   15433       2846         1723

Fine-tuned vs Base: +-5% correctness | +12% relevancy | 89% faster

Model: Llama 3.2 3B | LoRA rank-16 | 120 train / 30 test examples
Judge: GPT-4o-mini | Dataset: PatronusAI/FinanceBench
Bedrock model: us.meta.llama3-2-3b-instruct-v1:0

Full results saved to Drive ✓


In [19]:
# Side-by-side sample comparison for 5 questions
# Useful for GitHub README or portfolio write-up

print("SAMPLE ANSWER COMPARISON (5 questions)")
print("=" * 90)

for i in range(min(5, len(base_results))):
    b  = base_results[i]
    f  = ft_results[i]
    br = bedrock_results[i]
    print(f"\nQ{i+1}: {b['question'][:100]}...")
    print(f"  Ground Truth : {b['ground_truth']}")
    print(f"  Base (local) : {b['base_answer'][:100]}")
    print(f"  Bedrock      : {br['bedrock_answer'][:100]}")
    print(f"  Fine-tuned   : {f['ft_answer'][:100]}")
    print(f"  Correctness  : Base={base_correctness[i]:.2f} | Bedrock={bedrock_correctness[i]:.2f} | Fine-tuned={ft_correctness[i]:.2f}")
    print("-" * 90)

SAMPLE ANSWER COMPARISON (5 questions)

Q1: Based on the information provided primarily in the balance sheet and the statement of income, what i...
  Ground Truth : 63.86
  Base (local) : I don't have direct access to Corning's financial statements. However, I can guide you through the s
  Bedrock      : To calculate the FY2020 Days Payable Outstanding (DPO) for Corning, we need to gather the necessary 
  Fine-tuned   : 0.00
  Correctness  : Base=0.00 | Bedrock=0.00 | Fine-tuned=0.00
------------------------------------------------------------------------------------------

Q2: Does Foot Locker's new CEO have previous CEO experience in a similar company to Footlocker?...
  Ground Truth : Yes. She was previous CEO of Ulta Beauty which means she had to manage a large retail company that has brick and mortar + online business. So yes she was a CEO in a similar company to Foot Locker before this.
  Base (local) : I couldn't find any information on a new CEO of Foot Locker. However, I can p